In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from os.path import join
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score

pd.set_option('display.max_colwidth', 255)

In [ ]:
df = pd.read_parquet(join('predictions-single_prompt', 'rules_predictions.parquet'))

In [ ]:
df_annotations = pd.read_parquet(join('..','rule_manual_annotation_cleaned.parquet'))

In [76]:
for col in df.columns:
    print(col)

Unnamed: 0
name
joined_rules
translated_rules
be_nice
dont_be_dick
hate_speech
covenant
harassment
CP
zoophilia
NSFW
politics
religion
porn
violence
porn_nocw
violent_no_cw
nsfw_no_cw
spoilers_no_cw
sensitive_no_cw
politics_no_cw
cw_general
no_cw_general
bots_unmarked_no_permission
gov_accounts
far_right_extremist_nationalism_fascism
illegal
consequences_moderation
spam
incitement_violence
mis_disinfo_conspiracy_diffamation_propaganda
doxxing_personal_info
ads
images
brigading_dogpiling
behavior_content_format_allowed
codes_conduct
copyright_piracy_credit
server_interference_disruption
automated_tools_bots
topic_specific
language_specific
low_quality_content
reposting_crossposting
trolling
links_outside_content
min_age
spoilers
crypto
ai_content
need_use_real_name
reverse_systemic_oppression
scrape_model_training
private_no_reg
impersonation
purchase_mbship
block_report_mute
no_fediblock
freedom
anti_capitalist
public_order_morals
no_rule
gpt5-advertising
gpt5-ai_generated
gpt5-animal_

In [ ]:
rename_map = {
    'porn': 'adult_content',
    'porn_nocw': 'adult_content_cw',
    'NSFW': 'nsfw',
    'nsfw_no_cw': 'nsfw_cw',
    'CP': 'CSAM',
    'zoophilia': 'animal_cruelty',
    'doxxing_personal_info': 'privacy',
    'violent_no_cw': 'violence_cw',
    'politics_no_cw': 'politics_cw',
    'ads': 'advertising',
    'mis_disinfo_conspiracy_diffamation_propaganda': 'misinfo',
    'far_right_extremist_nationalism_fascism': 'nazis',
    'ai_content': 'ai_generated',
    'dick': 'dont_be_dick',
    'spoilers_no_cw': 'spoilers_cw',
    'sensitive_no_cw': 'sensitive_cw',
}

In [ ]:
df = df.rename(columns=rename_map)
df_updated_annotations = df_annotations.rename(columns=rename_map)

In [91]:
df.columns

Index(['Unnamed: 0', 'name', 'joined_rules', 'translated_rules', 'be_nice',
       'dont_be_dick', 'hate_speech', 'covenant', 'harassment', 'CSAM',
       ...
       'gpt5-registration_requirements', 'gpt5-religion',
       'gpt5-scrape_model_training', 'gpt5-sensitive_cw', 'gpt5-spam',
       'gpt5-spoilers', 'gpt5-spoilers_cw', 'gpt5-violence',
       'gpt5-violence_cw', 'gpt5-impersonation_note'],
      dtype='object', length=102)

In [ ]:
df_annotations.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'name', 'joined_rules',
       'translated_rules', 'be_nice', 'dont_be_dick', 'hate_speech',
       'covenant', 'harassment', 'CSAM', 'animal_cruelty', 'nsfw', 'politics',
       'religion', 'adult_content', 'violence', 'adult_content_cw',
       'violence_cw', 'nsfw_cw', 'spoilers_no_cw', 'sensitive_no_cw',
       'politics_cw', 'cw_general', 'no_cw_general',
       'bots_unmarked_no_permission', 'gov_accounts', 'nazis', 'illegal',
       'consequences_moderation', 'spam', 'incitement_violence', 'misinfo',
       'privacy', 'advertising', 'images', 'brigading_dogpiling',
       'behavior_content_format_allowed', 'codes_conduct',
       'copyright_piracy_credit', 'server_interference_disruption',
       'automated_tools_bots', 'topic_specific', 'language_specific',
       'low_quality_content', 'reposting_crossposting', 'trolling',
       'links_outside_content', 'min_age', 'spoilers', 'crypto',
       'ai_generated', 'need_use_real_name', 'reverse_

In [ ]:
df_annotations.to_parquet('predictions_for_linear_regression.parquet')

In [ ]:
if 'name' in df_annotations.columns and 'name' in df.columns:
    if df_annotations['name'].equals(df['name']):
        shared_cols = df_annotations.columns.intersection(df.columns)
        df.loc[:, shared_cols] = df_annotations[shared_cols]

In [82]:
df = df.fillna(0)

In [83]:
df = df.replace(r'^\s*$', 0, regex=True)

In [87]:
for col in df.columns:
    print(col)

Unnamed: 0
name
joined_rules
translated_rules
be_nice
dont_be_dick
hate_speech
covenant
harassment
CSAM
animal_cruelty
nsfw
politics
religion
adult_content
violence
adult_content_cw
violence_cw
nsfw_cw
spoilers_no_cw
sensitive_no_cw
politics_cw
cw_general
no_cw_general
bots_unmarked_no_permission
gov_accounts
nazis
illegal
consequences_moderation
spam
incitement_violence
misinfo
privacy
advertising
images
brigading_dogpiling
behavior_content_format_allowed
codes_conduct
copyright_piracy_credit
server_interference_disruption
automated_tools_bots
topic_specific
language_specific
low_quality_content
reposting_crossposting
trolling
links_outside_content
min_age
spoilers
crypto
ai_generated
need_use_real_name
reverse_systemic_oppression
scrape_model_training
private_no_reg
impersonation
purchase_mbship
block_report_mute
no_fediblock
freedom
anti_capitalist
public_order_morals
no_rule
gpt5-advertising
gpt5-ai_generated
gpt5-animal_cruelty
gpt5-automated_tools_bots
gpt5-be_nice
gpt5-block_rep

In [95]:
true_cols = [
    col for col in df.columns
    if col not in ["name", "joined_rules", "translated_rules", "Unnamed: 0"]
    and not col.startswith("gpt5-")
]

final_categories = [
    col for col in true_cols
    if f"gpt5-{col}" in df.columns
]

y_true_all = []
y_pred_all = []

df_f1 = []

for cat in final_categories:
    true = df[cat].astype(int)
    pred = df[f"gpt5-{cat}"].astype(int)

    y_true_all.extend(true)
    y_pred_all.extend(pred)

    report = classification_report(true, pred, output_dict=True)

    if "1" in report:
        f1_1 = report["1"]["f1-score"]
    else:
        f1_1 = 0.0
        
    df_f1.append({
        "category": cat,
        "F1_GPT-5-mini_experiment": round(f1_1 * 100, 2)
    })

micro_f1 = round(f1_score(y_true_all, y_pred_all, average='binary') * 100, 2)
macro_f1 = round(f1_score(y_true_all, y_pred_all, average='binary') * 100, 2)

print("Micro F1:", micro_f1)
print("Macro F1:", macro_f1)

df_f1 = pd.DataFrame(df_f1).sort_values("category").reset_index(drop=True)
print(df_f1)

Micro F1: 84.47
Macro F1: 84.47
                   category  F1_GPT-5-mini_experiment
0                      CSAM                     89.66
1             adult_content                     90.32
2          adult_content_cw                     82.96
3               advertising                     91.28
4              ai_generated                     85.71
5            animal_cruelty                     80.00
6      automated_tools_bots                     96.91
7                   be_nice                     88.14
8         block_report_mute                     83.64
9             codes_conduct                     65.28
10  consequences_moderation                     75.47
11  copyright_piracy_credit                     88.64
12                   crypto                     90.48
13               cw_general                     53.66
14             dont_be_dick                     60.00
15               harassment                     78.03
16              hate_speech                     97

In [85]:
df[df['gpt5-dont_be_dick']==1].shape

(163, 102)

In [86]:
#cw_cols = [c for c in df.columns if "cw" in c]
j = 1
for i, row in df[(df['gpt5-violence']==0) & (df['violence']==1)].iterrows():
    print(j)
    print(row['translated_rules'])
#    for col in cw_cols:
#        print(f"{col}: {row[col]}")
    print('***')
    j+=1

1
No racism, sexism, transphobia, or other discrimination against minorities no illegal
 no harassment, doxinization, spreading false information or other occurrence
 No spam no commercial activity is not pornographic, sexual or superior to violent content.
***
2
sexually explicit or violent media must be marked as sensitive when posting. no spam or advertising. no racism, sexism, homophobia, transphobia, xenophobia, or casteism. no incitement of violence or promotion of violent ideologies. no harassment, dogpiling or doxxing of other users. no illegal content. please understand - you have no privacy on this server or anywhere on mastodon. it's not designed to be private. assume that anything you post here, including dms, are publicly visible. no fake news, disinformation or misinformation.
***
3
nsfw posts (text and/or images) must be marked as such bigotry is not tolerated: no racism, sexism, homophobia, transphobia, xenophobia, or casteism no incitement of violence or promotion of v